# Distancia de Levenshtein

Es el mínimo número de inserciones, borrados y sustituciones de un carácter para convertir una cadena en otra. $D[i][j]$ es la distancia entre los prefijos $a_{1..i}$ y $b_{1..j}$: $D[i][0] = i$, $D[0][j] = j$ y $D[i][j] = \min(D[i-1][j] + 1,\ D[i][j-1] + 1,\ D[i-1][j-1] + [a_i \ne b_j])$. Cuesta $O(n \cdot m)$.

## 1. Distancia de edición con tabla

Calcular la tabla para `casa` y `calle`, mostrarla y calcular la distancia entre varias parejas de palabras.

In [1]:
def tabla_levenshtein(a, b):
    D = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        D[i][0] = i
    for j in range(len(b) + 1):
        D[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            D[i][j] = min(D[i - 1][j] + 1, D[i][j - 1] + 1, D[i - 1][j - 1] + (a[i - 1] != b[j - 1]))
    return D

def levenshtein(a, b):
    return tabla_levenshtein(a, b)[-1][-1]

a, b = "casa", "calle"
print("    " + " ".join(f"{x:>2}" for x in " " + b))
for letra, fila in zip(" " + a, tabla_levenshtein(a, b)):
    print(f"{letra:>2}  " + " ".join(f"{x:>2}" for x in fila))

for x, y in (("gato", "pato"), ("kitten", "sitting"), ("", "hola"), ("algoritmo", "logaritmo"), ("intención", "ejecución")):
    print(f"{x!r} → {y!r}: {levenshtein(x, y)}")

        c  a  l  l  e
     0  1  2  3  4  5
 c   1  0  1  2  3  4
 a   2  1  0  1  2  3
 s   3  2  1  1  2  3
 a   4  3  2  2  2  3
'gato' → 'pato': 1
'kitten' → 'sitting': 3
'' → 'hola': 4
'algoritmo' → 'logaritmo': 3
'intención' → 'ejecución': 5


## 2. Reconstruir las operaciones

Recorrer la tabla desde la esquina eligiendo en cada paso la casilla de la que sale el mínimo, y aplicar las operaciones una a una mostrando cómo cambia la palabra. Comprobar con palabras aleatorias que se llega a la segunda y que el número de operaciones es la distancia.

In [2]:
import random

def operaciones(a, b):
    D = tabla_levenshtein(a, b)
    i, j, ops = len(a), len(b), []
    while i or j:
        if i and j and D[i][j] == D[i - 1][j - 1] + (a[i - 1] != b[j - 1]):
            ops.append(("mantener" if a[i - 1] == b[j - 1] else "sustituir", a[i - 1], b[j - 1]))
            i, j = i - 1, j - 1
        elif i and D[i][j] == D[i - 1][j] + 1:
            ops.append(("borrar", a[i - 1], ""))
            i -= 1
        else:
            ops.append(("insertar", "", b[j - 1]))
            j -= 1
    return ops[::-1]

def aplica(a, ops, mostrar=False):
    palabra, pos = list(a), 0
    for op, viejo, nuevo in ops:
        if op == "borrar":
            del palabra[pos]
        elif op == "insertar":
            palabra.insert(pos, nuevo)
        else:
            palabra[pos] = nuevo
        if op != "borrar":
            pos += 1
        if mostrar and op != "mantener":
            detalle = f"{viejo} por {nuevo}" if op == "sustituir" else viejo or nuevo
            print(f"{op:<9} {detalle:<6} → {''.join(palabra)}")
    return "".join(palabra)

print("intención")
aplica("intención", operaciones("intención", "ejecución"), mostrar=True)

random.seed(2)
for _ in range(300):
    x = "".join(random.choices("abc", k=random.randint(0, 8)))
    y = "".join(random.choices("abc", k=random.randint(0, 8)))
    ops = operaciones(x, y)
    assert aplica(x, ops) == y
    assert sum(op != "mantener" for op, _, _ in ops) == levenshtein(x, y)

intención
sustituir i por e → entención
sustituir n por j → ejtención
sustituir t por e → ejeención
sustituir e por c → ejecnción
sustituir n por u → ejecución


## 3. Versión con dos filas

Cada fila de la tabla solo depende de la anterior, así que basta con guardar dos filas: memoria $O(\min(n, m))$. Comprobarla contra la tabla y contra una búsqueda en anchura que aplica operaciones sueltas hasta llegar a la otra cadena.

In [3]:
from collections import deque

def levenshtein_dos_filas(a, b):
    if len(a) < len(b):
        a, b = b, a
    anterior = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        actual = [i]
        for j, y in enumerate(b, 1):
            actual.append(min(anterior[j] + 1, actual[j - 1] + 1, anterior[j - 1] + (x != y)))
        anterior = actual
    return anterior[-1]

def levenshtein_bfs(a, b, alfabeto="abc"):
    # basta con borrar primero e insertar al final: nunca se pasa de la longitud máxima
    limite = max(len(a), len(b))
    dist, cola = {a: 0}, deque([a])
    while cola:
        s = cola.popleft()
        if s == b:
            return dist[s]
        vecinos = [s[:i] + s[i + 1:] for i in range(len(s))]
        vecinos += [s[:i] + x + s[i + 1:] for i in range(len(s)) for x in alfabeto]
        if len(s) < limite:
            vecinos += [s[:i] + x + s[i:] for i in range(len(s) + 1) for x in alfabeto]
        for v in vecinos:
            if v not in dist:
                dist[v] = dist[s] + 1
                cola.append(v)

print(levenshtein_dos_filas("intención", "ejecución"), levenshtein_dos_filas("abc", "abcabcabc"))

random.seed(3)
for _ in range(200):
    x = "".join(random.choices("abc", k=random.randint(0, 5)))
    y = "".join(random.choices("abc", k=random.randint(0, 5)))
    assert levenshtein_dos_filas(x, y) == levenshtein(x, y) == levenshtein_bfs(x, y)
for _ in range(200):
    x = "".join(random.choices("abcd", k=random.randint(0, 40)))
    y = "".join(random.choices("abcd", k=random.randint(0, 40)))
    assert levenshtein_dos_filas(x, y) == levenshtein(x, y)

5 6


## 4. Corrector ortográfico

Para una palabra que no está en el diccionario, sugerir las del diccionario a menor distancia (como mucho 2). La versión con dos filas puede abandonar en cuanto el mínimo de una fila supera el límite, porque ese mínimo nunca baja en las filas siguientes.

In [4]:
DICCIONARIO = """
algoritmo programa función variable recursividad iteración bucle lista tupla diccionario conjunto cadena
entero decimal booleano clase objeto método atributo herencia módulo paquete excepción archivo grafo árbol
nodo arista camino distancia matriz vector pila cola montículo búsqueda ordenación memoria tabla índice
clave valor python compilador intérprete sintaxis depurador prueba
""".split()

def levenshtein_acotada(a, b, limite):
    anterior = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        actual = [i]
        for j, y in enumerate(b, 1):
            actual.append(min(anterior[j] + 1, actual[j - 1] + 1, anterior[j - 1] + (x != y)))
        if min(actual) > limite:
            return None
        anterior = actual
    return anterior[-1] if anterior[-1] <= limite else None

def sugerencias(palabra, diccionario=DICCIONARIO, limite=2):
    if palabra in diccionario:
        return [palabra]
    distancias = {p: d for p in diccionario if (d := levenshtein_acotada(palabra, p, limite)) is not None}
    if not distancias:
        return []
    mejor = min(distancias.values())
    return sorted(p for p, d in distancias.items() if d == mejor)

for palabra in ("algoritmo", "algorimto", "funcion", "recursibidad", "pyton", "arbol", "matris", "kola", "zzzz"):
    print(f"{palabra:>13} → {sugerencias(palabra)}")

random.seed(4)
for _ in range(500):
    x = "".join(random.choices("abc", k=random.randint(0, 8)))
    y = "".join(random.choices("abc", k=random.randint(0, 8)))
    d = levenshtein(x, y)
    assert levenshtein_acotada(x, y, 2) == (d if d <= 2 else None)

    algoritmo → ['algoritmo']
    algorimto → ['algoritmo']
      funcion → ['función']
 recursibidad → ['recursividad']
        pyton → ['python']
        arbol → ['árbol']
       matris → ['matriz']
         kola → ['cola']
         zzzz → []
